# IVS (Atlas / Ipea)

Monta o painel municipal do Índice de Vulnerabilidade Social.
No livro publicado, mantenha os totais municipais: Total Cor, Total Sexo e Total Situação de Domicílio.
As definições das variáveis estão em **dictionary.csv**.


## Importações


In [6]:
import re
import numpy as np
import pandas as pd

from pathlib import Path
from typing import List, Optional
from unidecode import unidecode


## Configuração

In [7]:
def resolve_repo_root() -> Path:
    """Localiza a raiz do projeto que contém o diretório local de dados."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "books").is_dir():
            return candidate
    return start


BASE_DIR = resolve_repo_root() / "data"

RAW_IVS_DIR = BASE_DIR / "raw" / "ivs"
TRUSTED_IVS_DIR = BASE_DIR / "trusted" / "ivs"
TRUSTED_IBGE_DIR = BASE_DIR / "trusted" / "ibge"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_IVS_DIR, TRUSTED_IVS_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Leitura

In [8]:
path_ivs = RAW_IVS_DIR / "atlasivs_dadosbrutos_pt_v2.xlsx"


In [9]:
def ler_arquivo_ivs(path_arquivo: Path) -> pd.DataFrame:
    """
    Lê o arquivo AVS/IVS baixado manualmente da Base dos Dados.

    Suporta:
    - .csv
    - .csv.gz
    - .parquet
    - .xlsx
    - .xls
    """
    path_arquivo = Path(path_arquivo)

    if not path_arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path_arquivo}")

    suffixes = "".join(path_arquivo.suffixes).lower()

    if suffixes.endswith(".parquet"):
        df = pd.read_parquet(path_arquivo)

    elif suffixes.endswith(".csv.gz"):
        df = pd.read_csv(path_arquivo, dtype="string", compression="gzip")

    elif suffixes.endswith(".csv"):
        try:
            df = pd.read_csv(path_arquivo, dtype="string", encoding="utf-8")

            if df.shape[1] == 1:
                df = pd.read_csv(path_arquivo, sep=";", dtype="string", encoding="utf-8")

        except UnicodeDecodeError:
            df = pd.read_csv(path_arquivo, dtype="string", encoding="latin1")

            if df.shape[1] == 1:
                df = pd.read_csv(path_arquivo, sep=";", dtype="string", encoding="latin1")

    elif suffixes.endswith(".xlsx") or suffixes.endswith(".xls"):
        df = pd.read_excel(path_arquivo, dtype="string")

    else:
        raise ValueError(f"Extensão não suportada: {suffixes}")

    return df


In [10]:
df_ivs_raw = ler_arquivo_ivs(path_ivs)

print(df_ivs_raw.shape)
df_ivs_raw.head()


(340786, 103)


,id,nivel,ano,regiao,nome_regiao,outros_territs,uf,nome_uf,rm,nome_rm,...,t_scarteira_18m,t_setorpublico_18m,t_contapropria_18m,t_empregador_18m,t_formal_18m,t_fundc_ocup18m,t_medioc_ocup18m,t_supec_ocup18m,t_renda_todos_trabalhos,t_nremunerado_18m
0,1,brasil,2000,geral,-,-,geral,-,geral,-,...,"15,37","6,22","24,66","4,36","57,36","54,96","33,94","11,59","1585,45","4,93"
1,2,brasil,2010,geral,-,-,geral,-,geral,-,...,"15,55","6,12","22,81","3,09","65,13","69,67","53,13","19,07","1666,71","4,11"
2,3,brasil,2011,geral,-,-,geral,-,geral,-,...,"15,46","8,28","20,88","5,01","63,50","73,89","56,30","18,55","1434,40","2,64"
3,4,brasil,2012,geral,-,-,geral,-,geral,-,...,"15,31","8,62","20,32","5,52","64,65","74,09","58,36","19,78","1553,46","2,38"
4,5,brasil,2013,geral,-,-,geral,-,geral,-,...,"15,13","8,54","20,14","5,56","65,85","75,64","59,55","20,83","1599,55","2,13"


In [11]:
df_ivs_raw[
    (df_ivs_raw["label_cor"] == "Total Cor")&
    (df_ivs_raw["label_sexo"] == "Total Sexo")&
    (df_ivs_raw["label_sit_dom"] == "Total Situação de Domicílio")
    ].groupby("nome_regiao").agg({"nome_regiao": "count"})


,nome_regiao
nome_regiao,
-,76
CENTRO-OESTE,2493
NORDESTE,7724
NORTE,2078
SUDESTE,18365
SUL,5105


In [12]:
df_ivs_raw[
    # (df_ivs_raw["uf"] != "geral")&
    # (df_ivs_raw["nivel"] == "regiao,uf")&
    (df_ivs_raw["label_cor"] == "Total Cor")&
    (df_ivs_raw["label_sexo"] == "Total Sexo")&
    (df_ivs_raw["label_sit_dom"] == "Total Situação de Domicílio")
    ]


,id,nivel,ano,regiao,nome_regiao,outros_territs,uf,nome_uf,rm,nome_rm,...,t_scarteira_18m,t_setorpublico_18m,t_contapropria_18m,t_empregador_18m,t_formal_18m,t_fundc_ocup18m,t_medioc_ocup18m,t_supec_ocup18m,t_renda_todos_trabalhos,t_nremunerado_18m
21,22,brasil,2000,geral,-,-,geral,-,geral,-,...,"22,40","5,96","24,48","3,09","51,63","46,47","30,84","7,97","0,00","6,15"
22,23,brasil,2010,geral,-,-,geral,-,geral,-,...,"19,33","5,61","21,73","2,05","59,32","62,29","44,91","13,19","1296,19","5,58"
23,24,brasil,2011,geral,-,-,geral,-,geral,-,...,"18,99","7,43","21,59","3,53","56,74","67,35","47,63","12,82","1139,72","3,17"
24,25,brasil,2012,geral,-,-,geral,-,geral,-,...,"18,86","7,63","21,17","3,91","57,70","66,88","49,09","13,44","1215,64","2,88"
25,26,brasil,2013,geral,-,-,geral,-,geral,-,...,"18,29","7,63","21,06","3,87","58,68","68,32","50,15","14,21","1253,84","2,76"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340781,646600,"regiao,uf",2017,4,SUL,-,41,Paraná,geral,-,...,"14,61","11,06","24,98","5,42","68,58","74,43","55,89","19,57","1515,47","1,42"
340782,646606,"regiao,uf",2016,4,SUL,-,43,Rio Grande do Sul,geral,-,...,"13,51","11,55","23,61","6,12","71,53","72,99","53,78","16,95","1508,67","1,84"
340783,646607,"regiao,uf",2017,4,SUL,-,43,Rio Grande do Sul,geral,-,...,"13,52","11,86","24,02","6,52","70,87","73,19","54,22","17,74","1504,56","2,27"
340784,646613,"regiao,uf",2016,4,SUL,-,42,Santa Catarina,geral,-,...,"10,18","10,85","22,39","5,64","78,33","77,17","56,74","18,38","1484,16","0,62"


## Funções Auxiliares

In [13]:
def normalizar_nome_coluna(col: str) -> str:
    """
    Normaliza nomes de colunas:
    - remove acentos;
    - transforma em minúsculas;
    - substitui caracteres especiais por underscore.
    """
    col = str(col).strip()
    col = unidecode(col)
    col = col.lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")

    return col


def normalizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalizar_nome_coluna(col) for col in df.columns]
    return df


def normalizar_texto_valor(s: pd.Series) -> pd.Series:
    """
    Normaliza valores textuais para comparação.
    """
    return (
        s.astype("string")
        .fillna("")
        .map(lambda x: unidecode(str(x)).upper().strip())
        .str.replace(r"\s+", " ", regex=True)
    )


def parse_numero_br(s: pd.Series) -> pd.Series:
    """
    Converte números no padrão brasileiro.

    Exemplos:
        '0,512'   -> 0.512
        '488,75'  -> 488.75
        '20396'   -> 20396
        '-'       -> NaN
    """
    return (
        s.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "-": pd.NA,
            "...": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "NULL": pd.NA,
            "null": pd.NA,
        })
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )


## Padronizar Colunas

In [14]:
COLUNAS_ID_IVS_IPEA = [
    "regiao",
    "nome_regiao",
    "uf",
    "nome_uf",
    "rm",
    "nome_rm",
    "municipio",
    "municipio_6digt",
    "nome_municipio_uf",
    "udh",
    "nome_udh",
    "ano",
    "label_cor",
    "label_sexo",
    "label_sit_dom",
]

COLUNAS_CATEGORICAS_IVS_IPEA = [
    "prosp_soc",
]


## Filtrar Públicos Totais

In [15]:
df_ivs_total = df_ivs_raw[
    # (df_ivs_raw["uf"] != "geral")&
    # (df_ivs_raw["nivel"] == "regiao,uf")&
    (df_ivs_raw["label_cor"] == "Total Cor")&
    (df_ivs_raw["label_sexo"] == "Total Sexo")&
    (df_ivs_raw["label_sit_dom"] == "Total Situação de Domicílio")
    ]


In [16]:
print(df_ivs_raw.shape, df_ivs_total.shape)


(340786, 103) (35841, 103)


## Padronizar IVS/Ipea

In [17]:
def agregar_ivs_duplicado_municipio_ano(df: pd.DataFrame) -> pd.DataFrame:
    """
    Consolida duplicidades por município-ano.

    Útil caso a base tenha múltiplas UDHs por município.
    """
    df = df.copy()

    chaves = ["codigo_ibge", "ano"]

    cols_id = [
        col for col in df.columns
        if col.endswith("_origem")
        or col in ["codigo_ibge", "codigo_ibge_6", "ano", "prosp_soc"]
    ]

    cols_num = [
        col for col in df.columns
        if col not in cols_id
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    cols_pop_soma = [
        col for col in cols_num
        if (
            col == "populacao"
            or col.startswith("pop")
            or col.startswith("pea")
            or col in [
                "vulner15a24",
                "mchefe_fmenor",
                "vulner_dia",
                "dom_vulner_idoso",
            ]
        )
    ]

    cols_media = [col for col in cols_num if col not in cols_pop_soma]

    partes = []

    for (codigo_ibge, ano), g in df.groupby(chaves, dropna=False):
        row = {
            "codigo_ibge": codigo_ibge,
            "ano": ano,
        }

        if "codigo_ibge_6" in g.columns:
            row["codigo_ibge_6"] = g["codigo_ibge_6"].dropna().iloc[0] if g["codigo_ibge_6"].notna().any() else pd.NA

        for col in [c for c in cols_id if c not in row and c not in chaves]:
            if col in g.columns:
                valores = g[col].dropna()
                row[col] = valores.mode().iloc[0] if not valores.mode().empty else (valores.iloc[0] if len(valores) else pd.NA)

        for col in cols_pop_soma:
            row[col] = g[col].sum(skipna=True)

        if "populacao" in g.columns and g["populacao"].notna().sum() > 0 and g["populacao"].sum() > 0:
            peso = g["populacao"]

            for col in cols_media:
                mask = g[col].notna() & peso.notna() & (peso > 0)

                if mask.sum() > 0:
                    row[col] = np.average(g.loc[mask, col], weights=peso.loc[mask])
                else:
                    row[col] = np.nan
        else:
            for col in cols_media:
                row[col] = g[col].mean(skipna=True)

        partes.append(row)

    out = pd.DataFrame(partes)

    return out


In [18]:
def padronizar_ivs_ipea_municipio_ano(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza a base IVS/Ipea para granularidade município-ano.

    Usa:
        municipio -> codigo_ibge de 7 dígitos
        municipio_6digt -> codigo_ibge_6

    Retorna:
        codigo_ibge
        codigo_ibge_6
        ano
        variáveis numéricas do IVS/Ipea
        variáveis categóricas relevantes
    """
    df = normalizar_colunas(df_raw)

    colunas_obrigatorias = [
        "municipio",
        "municipio_6digt",
        "ano",
    ]

    for col in colunas_obrigatorias:
        if col not in df.columns:
            raise ValueError(
                f"Coluna obrigatória ausente: {col}. "
                f"Colunas disponíveis: {df.columns.tolist()}"
            )

    out = pd.DataFrame()

    out["codigo_ibge"] = (
        df["municipio"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out["codigo_ibge_6"] = (
        df["municipio_6digt"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(6)
    )

    out["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")

    # Campos úteis para auditoria
    for col in [
        "regiao",
        "nome_regiao",
        "uf",
        "nome_uf",
        "rm",
        "nome_rm",
        "nome_municipio_uf",
        "udh",
        "nome_udh",
        "label_cor",
        "label_sexo",
        "label_sit_dom",
    ]:
        if col in df.columns:
            out[f"{col}_origem"] = df[col].astype("string")

    # Colunas que não devem virar numéricas
    colunas_excluir = set(COLUNAS_ID_IVS_IPEA + COLUNAS_CATEGORICAS_IVS_IPEA)

    # Variáveis categóricas
    if "prosp_soc" in df.columns:
        out["prosp_soc"] = df["prosp_soc"].astype("string")

    # Todas as demais colunas candidatas a numéricas
    colunas_candidatas_numericas = [
        col for col in df.columns
        if col not in colunas_excluir
    ]

    for col in colunas_candidatas_numericas:
        serie_num = parse_numero_br(df[col])

        if serie_num.notna().sum() > 0:
            out[col] = serie_num

    # Remove linhas inválidas
    out = out.dropna(subset=["codigo_ibge", "ano"])

    # Caso ainda exista duplicidade município-ano, consolida.
    # Isso pode acontecer se houver mais de uma UDH para a mesma cidade.
    duplicadas = out.duplicated(["codigo_ibge", "ano"]).sum()

    if duplicadas > 0:
        print(f"Atenção: existem {duplicadas:,} linhas duplicadas por codigo_ibge-ano.")
        print("Será feita consolidação por município-ano.")

        out = agregar_ivs_duplicado_municipio_ano(out)

    out = (
        out
        .sort_values(["codigo_ibge", "ano"])
        .reset_index(drop=True)
    )

    return out


In [19]:
ivs_municipio_ano = padronizar_ivs_ipea_municipio_ano(df_ivs_total)

print(ivs_municipio_ano.shape)


Atenção: existem 24,697 linhas duplicadas por codigo_ibge-ano.
Será feita consolidação por município-ano.
(11144, 101)


In [20]:
ivs_municipio_ano = ivs_municipio_ano[ivs_municipio_ano["codigo_ibge"] != "0000000"]
ivs_municipio_ano.head()


,codigo_ibge,ano,codigo_ibge_6,regiao_origem,nome_regiao_origem,uf_origem,nome_uf_origem,rm_origem,nome_rm_origem,nome_municipio_uf_origem,...,t_scarteira_18m,t_setorpublico_18m,t_contapropria_18m,t_empregador_18m,t_formal_18m,t_fundc_ocup18m,t_medioc_ocup18m,t_supec_ocup18m,t_renda_todos_trabalhos,t_nremunerado_18m
14,1100015,2000,110001,1,NORTE,11,Rondônia,-,-,Alta Floresta D'Oeste (RO),...,20.48,2.46,47.37,1.83,22.83,22.51,11.60,1.38,0.00,12.51
15,1100015,2010,110001,1,NORTE,11,Rondônia,-,-,Alta Floresta D'Oeste (RO),...,23.17,5.31,34.73,2.83,41.68,45.49,31.05,7.72,950.08,9.63
16,1100023,2000,110002,1,NORTE,11,Rondônia,-,-,Ariquemes (RO),...,23.03,5.38,30.90,4.27,41.42,36.12,20.64,3.91,0.00,6.05
17,1100023,2010,110002,1,NORTE,11,Rondônia,-,-,Ariquemes (RO),...,19.21,6.30,27.66,2.30,52.81,57.26,39.76,9.29,1274.11,4.10
18,1100031,2000,110003,1,NORTE,11,Rondônia,-,-,Cabixi (RO),...,27.90,7.39,34.47,4.03,25.50,20.67,10.53,0.83,0.00,14.03


In [21]:
ivs_municipio_ano[ivs_municipio_ano["codigo_ibge"] == "3509502"]


,codigo_ibge,ano,codigo_ibge_6,regiao_origem,nome_regiao_origem,uf_origem,nome_uf_origem,rm_origem,nome_rm_origem,nome_municipio_uf_origem,...,t_scarteira_18m,t_setorpublico_18m,t_contapropria_18m,t_empregador_18m,t_formal_18m,t_fundc_ocup18m,t_medioc_ocup18m,t_supec_ocup18m,t_renda_todos_trabalhos,t_nremunerado_18m
6762,3509502,2000,350950,3,SUDESTE,35,São Paulo,63502,RM Campinas,Campinas (SP),...,15.90807,4.138823,18.881865,3.966918,70.198878,63.501657,44.584643,15.660878,988.434699,1.006103
6763,3509502,2010,350950,3,SUDESTE,35,São Paulo,63502,RM Campinas,Campinas (SP),...,10.76114,3.539001,17.178871,2.647393,77.573558,74.601136,57.019442,22.994680,2009.805683,1.373596


In [22]:
ivs_municipio_ano[ivs_municipio_ano["codigo_ibge"] == "4118204"]


,codigo_ibge,ano,codigo_ibge_6,regiao_origem,nome_regiao_origem,uf_origem,nome_uf_origem,rm_origem,nome_rm_origem,nome_municipio_uf_origem,...,t_scarteira_18m,t_setorpublico_18m,t_contapropria_18m,t_empregador_18m,t_formal_18m,t_fundc_ocup18m,t_medioc_ocup18m,t_supec_ocup18m,t_renda_todos_trabalhos,t_nremunerado_18m
8340,4118204,2000,411820,4,SUL,41,Paraná,-,-,Paranaguá (PR),...,20.71,2.49,22.08,2.84,60.80,52.88,32.01,8.20,0.0,1.08
8341,4118204,2010,411820,4,SUL,41,Paraná,-,-,Paranaguá (PR),...,16.05,3.47,20.24,2.11,67.86,66.03,46.49,11.15,1356.2,2.51


In [23]:
(
    ivs_municipio_ano
    .groupby("ano")
    .agg(
        qtd_municipios=("codigo_ibge", "nunique"),
        ivs_medio=("ivs", "mean"),
        ivs_min=("ivs", "min"),
        ivs_max=("ivs", "max"),
        populacao_total=("populacao", "sum")
    )
    .reset_index()
    .sort_values("ano")
)


,ano,qtd_municipios,ivs_medio,ivs_min,ivs_max,populacao_total
0,2000,5565,435.285530,2.0,872.0,242427601
1,2010,5565,320.313017,2.0,784.0,273276433


In [24]:
ivs_municipio_ano.columns.to_list()


['codigo_ibge',
 'ano',
 'codigo_ibge_6',
 'regiao_origem',
 'nome_regiao_origem',
 'uf_origem',
 'nome_uf_origem',
 'rm_origem',
 'nome_rm_origem',
 'nome_municipio_uf_origem',
 'udh_origem',
 'nome_udh_origem',
 'label_cor_origem',
 'label_sexo_origem',
 'label_sit_dom_origem',
 'prosp_soc',
 'populacao',
 'vulner15a24',
 'mchefe_fmenor',
 'vulner_dia',
 'dom_vulner_idoso',
 'pop0a1',
 'pop1a3',
 'pop4',
 'pop5',
 'pop6',
 'pop6a10',
 'pop6a17',
 'pop11a13',
 'pop11a14',
 'pop12a14',
 'pop15m',
 'pop15a17',
 'pop15a24',
 'pop16a18',
 'pop18m',
 'pop18a20',
 'pop18a24',
 'pop19a21',
 'pop25m',
 'pop65m',
 'pea10m',
 'pea10a14',
 'pea15a17',
 'pea18m',
 'id',
 'ivs',
 'ivs_infraestrutura_urbana',
 'ivs_capital_humano',
 'ivs_renda_e_trabalho',
 'idhm',
 'idhm_long',
 'idhm_educ',
 'idhm_renda',
 'idhm_educ_sub_esc',
 'idhm_educ_sub_freq',
 't_sem_agua_esgoto',
 't_sem_lixo',
 't_vulner_mais1h',
 't_mort1',
 't_c0a5_fora',
 't_c6a14_fora',
 't_m10a17_filho',
 't_mchefe_fundin_fmenor',
 

In [25]:
ivs_municipio_ano.to_parquet(
    TRUSTED_IBGE_DIR / "ivs_municipio_ano.parquet",
    index=False
)


## Base Final - Funções

In [26]:
import re
import numpy as np
import pandas as pd

from typing import Optional, List


In [27]:
ivs_municipio_ano = pd.read_parquet(
    TRUSTED_IBGE_DIR / "ivs_municipio_ano.parquet"
)
ivs_municipio_ano.shape


(11130, 101)

In [28]:
def normalizar_codigo_municipio(series: pd.Series, tamanho: int = 7) -> pd.Series:
    """
    Normaliza códigos municipais evitando erro do tipo:
        1100015.0 -> 11000150

    Funciona para códigos em formato string, int ou float.
    """

    s = series.copy()

    num = pd.to_numeric(s, errors="coerce")

    out = pd.Series(pd.NA, index=s.index, dtype="string")

    mask_num = num.notna()

    out.loc[mask_num] = (
        num.loc[mask_num]
        .astype("Int64")
        .astype("string")
        .str.zfill(tamanho)
    )

    mask_str = ~mask_num

    out.loc[mask_str] = (
        s.loc[mask_str]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(tamanho)
    )

    out = out.replace({"": pd.NA})

    return out


In [29]:
def np_select_texto(condicoes, escolhas, index=None, default=pd.NA):
    """
    Versão segura do np.select para retornar categorias em texto.

    Evita erro de dtype em versões novas do numpy quando misturamos:
        escolhas = strings
        default = np.nan
    """

    marcador_missing = "__MISSING__"

    condicoes_limpas = []

    for cond in condicoes:
        cond_series = pd.Series(cond, index=index)
        cond_series = cond_series.fillna(False).astype(bool)
        condicoes_limpas.append(cond_series.to_numpy())

    arr = np.select(
        condicoes_limpas,
        escolhas,
        default=marcador_missing
    ).astype("object")

    serie = pd.Series(arr, index=index, dtype="object")
    serie = serie.replace(marcador_missing, default)

    return serie


In [30]:
def padronizar_chaves_ivs(
    df: pd.DataFrame,
    col_id: str = "codigo_ibge",
    col_ano: str = "ano"
) -> pd.DataFrame:
    """
    Padroniza chaves principais da base IVS/IDHM.
    """

    out = df.copy()

    out[col_id] = normalizar_codigo_municipio(out[col_id], tamanho=7)

    out[col_ano] = pd.to_numeric(
        out[col_ano],
        errors="coerce"
    ).astype("Int64")

    if "codigo_ibge_6" in out.columns:
        out["codigo_ibge_6"] = normalizar_codigo_municipio(
            out["codigo_ibge_6"],
            tamanho=6
        )

    return out


In [31]:
def identificar_colunas_numericas_ivs(
    df: pd.DataFrame,
    col_id: str = "codigo_ibge",
    col_ano: str = "ano",
    cols_excluir: Optional[List[str]] = None,
    min_share_convertivel: float = 0.80
) -> List[str]:
    """
    Identifica colunas numéricas para cálculo de variação 2000-2010.

    Exclui chaves, nomes, códigos, labels e variáveis categóricas.
    """

    if cols_excluir is None:
        cols_excluir = []

    cols_excluir_padrao = [
        col_id,
        col_ano,
        "codigo_ibge_6",
        "id",

        "regiao_origem",
        "nome_regiao_origem",
        "uf_origem",
        "nome_uf_origem",
        "rm_origem",
        "nome_rm_origem",
        "nome_municipio_uf_origem",
        "udh_origem",
        "nome_udh_origem",
        "label_cor_origem",
        "label_sexo_origem",
        "label_sit_dom_origem",

        # categórica
        "prosp_soc",
    ]

    cols_excluir_total = set(cols_excluir_padrao + cols_excluir)

    cols_numericas = []

    for col in df.columns:
        if col in cols_excluir_total:
            continue

        serie_num = pd.to_numeric(df[col], errors="coerce")
        share_convertivel = serie_num.notna().mean()

        if share_convertivel >= min_share_convertivel:
            cols_numericas.append(col)

    return cols_numericas


In [32]:
def classificar_sentido_indicador(col: str) -> Optional[str]:
    """
    Define se, para o indicador, maior valor representa melhora ou piora.

    Retorna:
        "maior_melhor"
        "menor_melhor"
        None
    """

    menor_melhor_exatos = {
        "ivs",
        "ivs_infraestrutura_urbana",
        "ivs_capital_humano",
        "ivs_renda_e_trabalho",

        "i_gini",

        "t_sem_agua_esgoto",
        "t_sem_lixo",
        "t_vulner_mais1h",
        "t_mort1",
        "t_c0a5_fora",
        "t_c6a14_fora",
        "t_m10a17_filho",
        "t_mchefe_fundin_fmenor",
        "t_analf_15m",
        "t_analf_18m",
        "t_analf_25m",
        "t_cdom_fundin",
        "t_p15a24_nada",
        "t_vulner",
        "t_desocup18m",
        "t_p18m_fundin_informal",
        "t_vulner_depende_idosos",
        "t_atividade10a14",
        "t_fmor5",
        "t_razdep",
        "t_fectot",
        "t_env",
        "t_densidadem2",
        "t_scarteira_18m",
        "t_nremunerado_18m",
    }

    maior_melhor_exatos = {
        "idhm",
        "idhm_long",
        "idhm_educ",
        "idhm_renda",
        "idhm_educ_sub_esc",
        "idhm_educ_sub_freq",

        "espvida",
        "renda_per_capita",

        "t_pop18m_fundc",
        "t_pop5a6_escola",
        "t_pop11a13_ffun",
        "t_pop15a17_fundc",
        "t_pop18a20_medioc",

        "t_eletrica",
        "t_renda_trab",
        "t_carteira_18m",
        "t_formal_18m",
        "t_fundc_ocup18m",
        "t_medioc_ocup18m",
        "t_supec_ocup18m",
        "t_renda_todos_trabalhos",
    }

    if col in menor_melhor_exatos:
        return "menor_melhor"

    if col in maior_melhor_exatos:
        return "maior_melhor"

    return None


In [33]:
def consolidar_ivs_municipio_ano(
    df: pd.DataFrame,
    col_id: str = "codigo_ibge",
    col_ano: str = "ano",
    cols_numericas: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    Garante uma linha por município-ano.

    Para colunas numéricas, usa média.
    Para colunas não numéricas, usa o primeiro valor.
    """

    out = df.copy()

    if cols_numericas is None:
        cols_numericas = identificar_colunas_numericas_ivs(
            out,
            col_id=col_id,
            col_ano=col_ano
        )

    for col in cols_numericas:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    duplicadas = out.duplicated([col_id, col_ano]).sum()

    if duplicadas > 0:
        print(
            f"Atenção: foram encontradas {duplicadas} duplicidades por "
            f"{col_id}-{col_ano}."
        )
        print("Consolidando por média nas variáveis numéricas e primeiro valor nas demais.")

    cols_nao_numericas = [
        col for col in out.columns
        if col not in cols_numericas + [col_id, col_ano]
    ]

    agg_dict = {}

    for col in cols_nao_numericas:
        agg_dict[col] = "first"

    for col in cols_numericas:
        agg_dict[col] = "mean"

    out = (
        out
        .groupby([col_id, col_ano], as_index=False)
        .agg(agg_dict)
        .sort_values([col_id, col_ano])
        .reset_index(drop=True)
    )

    return out


In [34]:
def criar_abt_ivs_2010_com_variacao_2000_2010(
    df_ivs: pd.DataFrame,
    col_id: str = "codigo_ibge",
    col_ano: str = "ano",
    ano_base: int = 2000,
    ano_ref: int = 2010,
    cols_variacao: Optional[List[str]] = None,
    cols_excluir_variacao: Optional[List[str]] = None,
    criar_var_rel: bool = True,
    criar_delta_log: bool = True,
    criar_cagr: bool = True,
    criar_melhoria_social: bool = True
) -> pd.DataFrame:
    """
    Cria uma ABT municipal com uma linha por município referente a 2010,
    adicionando variáveis de variação entre 2000 e 2010.

    Para cada variável numérica selecionada, cria:
        variavel_2000
        variavel_var_abs_2000_2010
        variavel_direcao_2000_2010
        variavel_var_rel_2000_2010
        variavel_delta_log_2000_2010
        variavel_cagr_2000_2010

    Para indicadores com direção normativa conhecida, cria:
        variavel_melhoria_abs_2000_2010
        variavel_flag_melhorou_2000_2010
        variavel_status_melhoria_2000_2010
    """

    df = padronizar_chaves_ivs(
        df_ivs,
        col_id=col_id,
        col_ano=col_ano
    )

    if cols_variacao is None:
        cols_variacao = identificar_colunas_numericas_ivs(
            df,
            col_id=col_id,
            col_ano=col_ano,
            cols_excluir=cols_excluir_variacao
        )

    for col in cols_variacao:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = consolidar_ivs_municipio_ano(
        df,
        col_id=col_id,
        col_ano=col_ano,
        cols_numericas=cols_variacao
    )

    df_base = df[df[col_ano] == ano_base].copy()
    df_ref = df[df[col_ano] == ano_ref].copy()

    if df_base.empty:
        raise ValueError(f"Não encontrei linhas para ano_base={ano_base}.")

    if df_ref.empty:
        raise ValueError(f"Não encontrei linhas para ano_ref={ano_ref}.")

    out = df_ref.copy()

    df_base_aux = df_base[[col_id] + cols_variacao].copy()

    df_base_aux = df_base_aux.rename(
        columns={
            col: f"{col}_{ano_base}"
            for col in cols_variacao
        }
    )

    out = out.merge(
        df_base_aux,
        on=col_id,
        how="left"
    )

    n_anos = ano_ref - ano_base

    for col in cols_variacao:
        col_ref = col
        col_base = f"{col}_{ano_base}"

        var_abs_col = f"{col}_var_abs_{ano_base}_{ano_ref}"

        # Variação absoluta
        out[var_abs_col] = out[col_ref] - out[col_base]

        # Direção bruta: aumento, redução ou estabilidade
        out[f"{col}_direcao_{ano_base}_{ano_ref}"] = np_select_texto(
            condicoes=[
                out[var_abs_col] > 0,
                out[var_abs_col] < 0,
                out[var_abs_col] == 0,
            ],
            escolhas=[
                "aumento",
                "reducao",
                "estavel",
            ],
            index=out.index,
            default=pd.NA
        )

        # Variação relativa
        if criar_var_rel:
            out[f"{col}_var_rel_{ano_base}_{ano_ref}"] = np.where(
                out[col_base].notna() & (out[col_base] != 0),
                out[col_ref] / out[col_base] - 1,
                np.nan
            )

        # Delta log
        if criar_delta_log:
            out[f"{col}_delta_log_{ano_base}_{ano_ref}"] = np.where(
                (out[col_ref] > 0) & (out[col_base] > 0),
                np.log(out[col_ref]) - np.log(out[col_base]),
                np.nan
            )

        # CAGR
        if criar_cagr:
            out[f"{col}_cagr_{ano_base}_{ano_ref}"] = np.where(
                (out[col_ref] > 0) & (out[col_base] > 0),
                (out[col_ref] / out[col_base]) ** (1 / n_anos) - 1,
                np.nan
            )

        # Melhoria social
        if criar_melhoria_social:
            sentido = classificar_sentido_indicador(col)

            if sentido == "maior_melhor":
                out[f"{col}_melhoria_abs_{ano_base}_{ano_ref}"] = (
                    out[col_ref] - out[col_base]
                )

            elif sentido == "menor_melhor":
                out[f"{col}_melhoria_abs_{ano_base}_{ano_ref}"] = (
                    out[col_base] - out[col_ref]
                )

            else:
                continue

    out[col_ano] = ano_ref

    cols_inicio = [
        col_id,
        col_ano,
        "codigo_ibge_6",
        "nome_municipio_uf_origem",
        "uf_origem",
        "nome_uf_origem",
        "regiao_origem",
        "nome_regiao_origem",
    ]

    cols_inicio = [col for col in cols_inicio if col in out.columns]

    out = out[
        cols_inicio + [col for col in out.columns if col not in cols_inicio]
    ].copy()

    return out


## Base Final

In [35]:
abt_ivs_2010_var = criar_abt_ivs_2010_com_variacao_2000_2010(
    df_ivs=ivs_municipio_ano,
    col_id="codigo_ibge",
    col_ano="ano",
    ano_base=2000,
    ano_ref=2010,
    criar_var_rel=True,
    criar_delta_log=True,
    criar_cagr=True,
    criar_melhoria_social=True
)

abt_ivs_2010_var.head()


/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/var/folders/m0/t6n3_g756gl_knfkbb_t93980000gn/T/ipykernel_64150/2719704342.py:111: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many t

,codigo_ibge,ano,codigo_ibge_6,nome_municipio_uf_origem,uf_origem,nome_uf_origem,regiao_origem,nome_regiao_origem,rm_origem,nome_rm_origem,...,t_renda_todos_trabalhos_var_rel_2000_2010,t_renda_todos_trabalhos_delta_log_2000_2010,t_renda_todos_trabalhos_cagr_2000_2010,t_renda_todos_trabalhos_melhoria_abs_2000_2010,t_nremunerado_18m_var_abs_2000_2010,t_nremunerado_18m_direcao_2000_2010,t_nremunerado_18m_var_rel_2000_2010,t_nremunerado_18m_delta_log_2000_2010,t_nremunerado_18m_cagr_2000_2010,t_nremunerado_18m_melhoria_abs_2000_2010
0,1100015,2010,110001,Alta Floresta D'Oeste (RO),11,Rondônia,1,NORTE,-,-,...,NaN,NaN,NaN,950.08,-2.88,reducao,-0.230216,-0.261645,-0.025825,2.88
1,1100023,2010,110002,Ariquemes (RO),11,Rondônia,1,NORTE,-,-,...,NaN,NaN,NaN,1274.11,-1.95,reducao,-0.322314,-0.389071,-0.038160,1.95
2,1100031,2010,110003,Cabixi (RO),11,Rondônia,1,NORTE,-,-,...,NaN,NaN,NaN,827.86,-0.45,reducao,-0.032074,-0.032600,-0.003255,0.45
3,1100049,2010,110004,Cacoal (RO),11,Rondônia,1,NORTE,-,-,...,NaN,NaN,NaN,1277.37,-2.77,reducao,-0.258396,-0.298939,-0.029452,2.77
4,1100056,2010,110005,Cerejeiras (RO),11,Rondônia,1,NORTE,-,-,...,NaN,NaN,NaN,926.57,-1.64,reducao,-0.138865,-0.149504,-0.014839,1.64


In [36]:
print(abt_ivs_2010_var.shape)
print(abt_ivs_2010_var["codigo_ibge"].nunique())
print(abt_ivs_2010_var["ano"].value_counts(dropna=False))


(5565, 656)
5565
ano
2010    5565
Name: count, dtype: int64


In [37]:
cols_ipea = ["codigo_ibge"]+ [col for col in abt_ivs_2010_var.columns if col.startswith("ivs")]+ [col for col in abt_ivs_2010_var.columns if col.startswith("idhm")]+ [col for col in abt_ivs_2010_var.columns if col.startswith("renda")]

cols_ipea = [col for col in cols_ipea if col in abt_ivs_2010_var.columns]

abt_ivs_2010_var[cols_ipea].head()


,codigo_ibge,ivs,ivs_infraestrutura_urbana,ivs_capital_humano,ivs_renda_e_trabalho,ivs_2000,ivs_infraestrutura_urbana_2000,ivs_capital_humano_2000,ivs_renda_e_trabalho_2000,ivs_var_abs_2000_2010,...,idhm_educ_sub_freq_cagr_2000_2010,idhm_educ_sub_freq_melhoria_abs_2000_2010,renda_per_capita,renda_per_capita_2000,renda_per_capita_var_abs_2000_2010,renda_per_capita_direcao_2000_2010,renda_per_capita_var_rel_2000_2010,renda_per_capita_delta_log_2000_2010,renda_per_capita_cagr_2000_2010,renda_per_capita_melhoria_abs_2000_2010
0,1100015,376.0,0.306,0.450,0.373,582.0,0.584,0.629,0.532,-206.0,...,0.075773,0.326,476.99,371.15,105.84,aumento,0.285168,0.250889,0.025406,105.84
1,1100023,268.0,0.149,0.384,0.271,42.0,0.258,0.575,0.427,226.0,...,0.061131,0.290,689.95,530.87,159.08,aumento,0.299659,0.262102,0.026557,159.08
2,1100031,3.0,0.128,0.389,0.383,514.0,0.442,0.587,0.514,-511.0,...,0.069056,0.341,457.17,342.46,114.71,aumento,0.334959,0.288900,0.029311,114.71
3,1100049,255.0,0.111,0.329,0.324,404.0,0.251,0.523,0.439,-149.0,...,0.053116,0.282,738.06,456.63,281.43,aumento,0.616320,0.480152,0.049187,281.43
4,1100056,28.0,0.144,0.365,0.330,491.0,0.419,0.524,0.530,-463.0,...,0.060690,0.317,577.18,511.47,65.71,aumento,0.128473,0.120865,0.012160,65.71


In [38]:
abt_ivs_2010_var[cols_ipea].to_parquet(
    TRUSTED_IBGE_DIR / "abt_ivs_2010_var.parquet",
    index=False
)
